# WI Generator — WI ต้นฉบับ (มีรูป/อุปกรณ์) → Work Instruction (.docx)

อัปโหลดไฟล์ **WI ต้นฉบับ** ตั้งแต่ 1 ไฟล์ขึ้นไป (เอกสารที่มีเนื้อหาขั้นตอนปฏิบัติงาน พร้อมรูปภาพประกอบและรายการอุปกรณ์ในตัว เช่น `DLTWI01AEP_system.docx`, `DLTWI02OCR_system.docx`, `DLTWI03Hotstamp_system.docx`) และไฟล์ **Template** (แบบฟอร์ม Work Instruction เปล่า) แล้วรันทุกเซลล์ตามลำดับ (Runtime → Run all) จะได้ไฟล์ผลลัพธ์**เดียว**ที่รวมทุกระบบที่อัปโหลด มีรูปแบบเหมือน template ทุกประการ พร้อมดาวน์โหลดอัตโนมัติในเซลล์สุดท้าย

ถ้าอัปโหลดหลายไฟล์ (หลายระบบ/เครื่องจักร) หัวข้อ **5. ขั้นตอนการปฏิบัติงาน** ในเอกสารผลลัพธ์จะถูกแบ่งเป็นหัวข้อย่อยให้อัตโนมัติ ตามลำดับไฟล์ที่อัปโหลด เช่น `5.1 AEP system`, `5.2 OCR System`, `5.3 Hot stamp System` แต่ละหัวข้อย่อยตามด้วยตาราง PM (W1/M1/M3/Y1) ของระบบนั้น ๆ

โครงสร้างไฟล์ต้นฉบับที่รองรับต่อไฟล์ (ตามไฟล์ต้นแบบ DLT-WI01-AEP system):
- ตารางหัวเอกสาร (แถวเดียว) ที่มีคำว่า `SOP Number` / `Title`
- ตารางหัวข้อแบบ 1 เซลล์ (banner) ที่คั่นแต่ละหมวด: `วัตถุประสงค์`, `ผู้ปฏิบัติ`, `อุปกรณ์`, `เอกสารอ้างอิง`, `แผนผังการดำเนินงาน`, `วิธีปฏิบัติ`, `เอกสารที่เกี่ยวข้องและการเก็บบันทึก`
- ในหมวด `วิธีปฏิบัติ` ขั้นตอนจะถูกแบ่งเป็นช่วงด้วยข้อความหัวรอบ เช่น "การตรวจสอบรอบ 1 สัปดาห์ / 1 เดือน / 3 เดือน / 1 ปี" และมีรูปภาพพร้อมคำบรรยาย "รูปที่ N ..." แทรกอยู่ระหว่างขั้นตอน
- ขั้นตอนที่อยู่**ก่อน**หัวรอบแรก (เช่น การเตรียมงาน/ปิดเครื่องจักรก่อนตรวจเช็ค) ถือเป็นขั้นตอนเตรียมงานร่วมของระบบนั้น จะถูกแทรกซ้ำไว้บนสุดของทุกตาราง PM ของระบบนั้นในข้อ 5 ของเอกสารผลลัพธ์

โค้ดนี้จะจับคู่รูปภาพและอุปกรณ์แต่ละอย่างเข้ากับขั้นตอนที่เกี่ยวข้องด้วย **ฮิวริสติก** (จับคู่คำในคำบรรยายรูป/ชื่ออุปกรณ์กับข้อความขั้นตอน) ไม่ใช่ข้อมูลที่ยืนยันความถูกต้อง 100% — ควรให้ทีมงานตรวจทานผลลัพธ์ในคอลัมน์ "รูปภาพประกอบ" และ "อุปกรณ์/เครื่องมือที่ใช้" อีกครั้งก่อนใช้งานจริง

## 1. ติดตั้งไลบรารีที่ต้องใช้

In [ ]:
!pip install -q python-docx pythainlp

## 2. อัปโหลดไฟล์ WI ต้นฉบับ (เลือกได้หลายไฟล์) และไฟล์ Template

เมื่อขึ้นหน้าต่างเลือกไฟล์ WI ต้นฉบับ สามารถกด Ctrl/Shift ค้างเพื่อเลือกได้หลายไฟล์พร้อมกัน (1 ไฟล์ = 1 ระบบ/เครื่องจักร) ลำดับที่เลือกจะกลายเป็นลำดับหัวข้อย่อย 5.1, 5.2, 5.3, ... ในเอกสารผลลัพธ์

In [ ]:
from google.colab import files

print("เลือกไฟล์ WI ต้นฉบับ (.docx) -- เลือกได้หลายไฟล์ (1 ไฟล์ = 1 ระบบ) ...")
uploaded_source_wi = files.upload()
source_wi_paths = list(uploaded_source_wi.keys())

print("\nเลือกไฟล์ Template (.docx) ...")
uploaded_template = files.upload()
template_path = next(iter(uploaded_template))

print(f"\nWI ต้นฉบับ ({len(source_wi_paths)} ไฟล์):")
for p in source_wi_paths:
    print(f"  - {p}")
print(f"Template: {template_path}")

## 3. โค้ดดึงข้อมูลจากไฟล์ WI ต้นฉบับ (`extract`)

แตกต่างจาก Check Sheet แบบเดิม (ตารางเช็คลิสต์ล้วน) ไฟล์ต้นฉบับใหม่นี้เป็นเอกสารบรรยายขั้นตอนที่มีรูปภาพฝังอยู่จริง และมีตารางหัวข้อ `อุปกรณ์` แยกต่างหาก โค้ดนี้จะ:
1. อ่านหัวเอกสาร (`SOP Number`, `Title`) และหมวดต่าง ๆ (`วัตถุประสงค์`, `ผู้ปฏิบัติ`, `อุปกรณ์`, ฯลฯ) โดยตรวจจากตาราง banner แถวเดียว
2. แบ่งขั้นตอนในหมวด `วิธีปฏิบัติ` ออกเป็นกลุ่มตามหัวรอบ (W1/M1/M3/Y1) โดยขั้นตอนก่อนหัวรอบแรกจัดเป็นกลุ่ม "เตรียมงานร่วม"
3. จับคู่รูปภาพที่แทรกอยู่กับขั้นตอนที่เกี่ยวข้องที่สุดในกลุ่มเดียวกัน โดยเทียบคำในคำบรรยายรูป ("รูปที่ N ...") กับข้อความขั้นตอน (ใช้ pythainlp ตัดคำภาษาไทย + จับคำศัพท์ภาษาอังกฤษเป็นสัญญาณหลัก เพราะคำเทคนิคอย่าง Servo/Cover tool/Cooling unit มักสะกดตรงกันทั้งสองฝั่ง) — เทียบกับ**ขั้นตอนทั้งหมดในกลุ่ม** ไม่ใช่แค่ขั้นตอนที่ผ่านมาแล้ว เพราะบางครั้งรูปจะถูกวางไว้ก่อนขั้นตอนที่มันอธิบายจริง ๆ

In [ ]:
"""
Extraction + generation logic for the new-format source document
(narrative "วิธีปฏิบัติ" style, e.g. DLTWI01AEP_system.docx) that carries its
own embedded photos and an equipment list -- unlike the old Check Sheet
format, which only had a checklist table and a blank tools table.
"""
import copy
import io
import re
from pathlib import Path

import docx
from docx.oxml.ns import qn
from docx.shared import Cm
from docx.table import Table

from pythainlp.tokenize import word_tokenize
from pythainlp.corpus import thai_stopwords
from pythainlp.util import Trie

# ---------------------------------------------------------------------------
# 1. Extraction from the new-format source document
# ---------------------------------------------------------------------------

CYCLE_LABELS = {
    "W1": "รายสัปดาห์ (W1)",
    "M1": "รายเดือน (M1)",
    "M3": "ทุก 3 เดือน (M3)",
    "6M": "ทุก 6 เดือน (6M)",
    "Y1": "รายปี (Y1)",
    "Y3": "ทุก 3 ปี (Y3)",
}

SECTION_BANNERS = {
    "purpose": "วัตถุประสงค์",
    "responsible": "ผู้ปฏิบัติ",
    "equipment": "อุปกรณ์",
    "references": "เอกสารอ้างอิง",
    "flow": "แผนผังการดำเนินงาน",
    "procedure": "วิธีปฏิบัติ",
    "records": "เอกสารที่เกี่ยวข้องและการเก็บบันทึก",
}

CYCLE_RE = re.compile(r"รอบ\s*(\d+)\s*(สัปดาห์|เดือน|ปี)")
CAPTION_START_RE = re.compile(r"^รูปที่\s*\d+")
DOC_CODE_RE = re.compile(r"([A-Za-z]{2,6}-WI\d+)")
UNIT_MAP = {"สัปดาห์": "W", "เดือน": "M", "ปี": "Y"}

_EXTRA_VOCAB = [
    "ไฮดรอลิค", "เซนเซอร์", "สวิตช์", "โหมด", "แม่พิมพ์", "ลูกปืน", "ตลับลูกปืน",
    "อ่างพักน้ำมัน", "ระดับน้ำมัน", "เกจวัดระดับ", "สายลม", "ข้อต่อ", "สเปรย์",
    "ผ้าสะอาด", "จาระบี", "แผงระบาย", "ปุ่ม", "น็อต", "สกรู", "ประแจ", "คราบ",
    "รอยรั่ว", "แท่นปั๊ม",
]
_STOP = set(thai_stopwords()) | {
    "ตรวจสอบ", "บันทึกผล", "จากนั้น", "หลังจาก", "ทำการ", "ทำ", "การ", "แล้ว",
    "หรือไม่", "ว่า", "เพื่อ", "ให้", "กับ", "ได้", "ไม่มี", "ไม่", "มี", "ปกติ",
    "การทำงาน", "เครื่องจักร", "ทั้งหมด", "หรือ", "จาก", "ใน", "ของ", "และ", "ที่",
    "เป็น", "อยู่", "ไป", "มา", "จะ", "ต้อง", "สามารถ", "ทุก", "อาจ",
}
_TRIE = Trie(list(thai_stopwords()) + _EXTRA_VOCAB)


def _thai_tokens(text):
    out = []
    for w in word_tokenize(text, engine="newmm", custom_dict=_TRIE):
        w = w.strip()
        if not w or w in _STOP or re.fullmatch(r"[\W\d]+", w):
            continue
        out.append(w.lower())
    return out


def _english_tokens(text):
    return set(t.lower() for t in re.findall(r"[A-Za-z]{3,}", text))


def _match_score(caption_text, step_text):
    th_overlap = sum(len(w) for w in (set(_thai_tokens(caption_text)) & set(_thai_tokens(step_text))))
    en_overlap = sum(len(w) for w in (_english_tokens(caption_text) & _english_tokens(step_text))) * 3
    return th_overlap + en_overlap


def _best_step_index(caption_text, steps, used, anchor_index):
    """Best-matching step for one photo's caption. Scored against the WHOLE
    step list of the group (not just steps seen so far) -- a photo's caption
    sometimes describes a step that appears a few lines AFTER the photo in
    the source document, not just the one immediately before it."""
    if not caption_text:
        return anchor_index if anchor_index is not None and anchor_index < len(steps) else None
    best_i, best_score = None, None
    for i, step in enumerate(steps):
        score = _match_score(caption_text, step["detail"])
        if i not in used:
            score += 0.5  # prefer spreading images across not-yet-illustrated steps
        if anchor_index is not None:
            score -= abs(i - anchor_index) * 0.05  # tie-break towards proximity
        if best_score is None or score > best_score:
            best_score, best_i = score, i
    return best_i


def _split_captions(text):
    parts = re.split(r"รูปที่\s*\d+", text)
    return [p.strip(" –-:\t") for p in parts if p.strip()]


def _cycle_code(number, unit):
    return f"{UNIT_MAP[unit]}{number}"


def _banner_key(tbl_elem):
    trs = tbl_elem.findall(qn("w:tr"))
    if len(trs) != 1:
        return None
    text = "".join(trs[0].xpath(".//w:t/text()")).strip()
    for key, keyword in SECTION_BANNERS.items():
        if text == keyword:
            return key
    return None


def _paragraph_images(p_elem, doc_part):
    out = []
    for blip in p_elem.findall(".//" + qn("a:blip")):
        rid = blip.get(qn("r:embed"))
        if rid and rid in doc_part.rels:
            out.append(doc_part.rels[rid].target_part.blob)
    return out


def _is_meaningful_step(text):
    stripped = re.sub(r"[\s%0-9.\-–—“”\"']", "", text)
    return len(stripped) >= 3


def _extract_banner_format(path):
    """Parse the narrative-style WI source document (see module docstring)."""
    doc = docx.Document(path)
    body = doc.element.body
    doc_part = doc.part

    meta = {"title": "", "doc_no": ""}
    purpose_lines, responsible_lines, equipment_list = [], [], []
    references_lines, records_lines = [], []
    groups = []  # [(cycle_code_or_None, [step, ...]), ...]
    current_steps = None

    if doc.tables:
        for row in doc.tables[0].rows:
            cells = [c.text.strip() for c in row.cells]
            if "SOP Number" in cells:
                i = cells.index("SOP Number")
                if i + 1 < len(cells):
                    meta["doc_no"] = cells[i + 1]
            if "Title" in cells:
                i = cells.index("Title")
                for c in cells[i + 1:]:
                    if c:
                        meta["title"] = c
                        break

    section = None
    pending_batch = []  # image blobs waiting for a caption line, within the current group
    current_events = []  # [(captions_list, blobs_list, anchor_index), ...] for the current group

    def close_group():
        """Resolve all pending photo captions against the group's FULL step
        list (a caption can describe a step that appears a few lines AFTER
        the photo in the source document, so matching needs the whole list,
        not just the steps seen so far)."""
        if pending_batch:
            anchor = len(current_steps) - 1 if current_steps else None
            current_events.append(([], list(pending_batch), anchor))
            pending_batch.clear()
        if current_steps:
            used = set()
            for captions, blobs, anchor in current_events:
                for idx, blob in enumerate(blobs):
                    cap_text = captions[idx] if idx < len(captions) else (captions[-1] if captions else "")
                    best_i = _best_step_index(cap_text, current_steps, used, anchor)
                    if best_i is not None:
                        current_steps[best_i]["images"].append(blob)
                        if cap_text:
                            current_steps[best_i].setdefault("captions", []).append(cap_text)
                        used.add(best_i)
        current_events.clear()

    for child in body.iterchildren():
        tag = child.tag.split("}")[-1]
        if tag == "tbl":
            close_group()
            banner = _banner_key(child)
            if banner:
                section = banner
                if banner == "procedure":
                    current_steps = []
                    groups.append((None, current_steps))
            continue
        if tag != "p":
            continue

        text = "".join(child.xpath(".//w:t/text()")).strip()
        images = _paragraph_images(child, doc_part)

        if section == "purpose" and text:
            purpose_lines.append(text)
        elif section == "responsible" and text:
            responsible_lines.append(text)
        elif section == "equipment" and text:
            equipment_list.append(text)
        elif section == "references" and text:
            references_lines.append(text)
        elif section == "records" and text:
            records_lines.append(text)
        elif section == "procedure":
            cyc = CYCLE_RE.search(text)
            if cyc and len(text) < 40 and not images:
                close_group()
                current_steps = []
                groups.append((_cycle_code(cyc.group(1), cyc.group(2)), current_steps))
                continue

            is_caption = bool(CAPTION_START_RE.match(text))
            if is_caption and pending_batch:
                captions = _split_captions(text)
                anchor = len(current_steps) - 1 if current_steps else None
                current_events.append((captions, list(pending_batch), anchor))
                pending_batch.clear()
                continue
            if is_caption:
                continue  # caption with nothing pending -- nothing to attribute it to

            if images and not _is_meaningful_step(text):
                pending_batch.extend(images)
                continue

            if text and _is_meaningful_step(text):
                if pending_batch:
                    anchor = len(current_steps) - 1 if current_steps else None
                    current_events.append(([], list(pending_batch), anchor))
                    pending_batch.clear()
                current_steps.append({"detail": re.sub(r"\s+", " ", text).strip(), "images": []})
                if images:
                    pending_batch.extend(images)
            elif images:
                pending_batch.extend(images)

    close_group()

    doc_code = ""
    for line in records_lines + references_lines + [meta["title"]]:
        m = DOC_CODE_RE.search(line)
        if m:
            doc_code = m.group(1)
            break

    subject = re.sub(r"(?i)preventive maintenance", "", meta["title"]).strip() or meta["title"]

    return {
        "meta": {"doc_no": meta["doc_no"] or doc_code, "subject": subject, "title": meta["title"]},
        "purpose_lines": purpose_lines,
        "responsible_lines": responsible_lines,
        "precaution_lines": [],
        "equipment_list": equipment_list,
        "groups": groups,
    }


# ---------------------------------------------------------------------------
# 1b. Extraction from the chapter-based format (e.g. MethodBASE01.docx):
# "บทที่ 1 ข้อกำหนดเบื้องต้น" / "บทที่ 2 รายละเอียดงาน", a personnel table,
# a tools/materials/equipment list, and a checklist table (ลำดับที่/คำอธิบาย)
# whose rows are enriched with the matching narrative paragraph + photos from
# the body. This format has no PM-cycle markers of its own -- every step goes
# into a single "6M" cycle, per instruction.
# ---------------------------------------------------------------------------

_CHAPTER_RE = re.compile(r"บทที่\s*(\d+)")
_STEP_HEADING_TRIM_RE = re.compile(r"^\s*\d+(\.\d+)*\s*")


def _norm_step_text(s):
    s = _STEP_HEADING_TRIM_RE.sub("", s)
    return re.sub(r"\s+", " ", s).strip().lower()


def _extract_chapter_format(path):
    doc = docx.Document(path)
    paragraphs = doc.paragraphs
    doc_part = doc.part

    # --- meta: doc number + subject from the cover lines ---
    meta = {"title": "", "doc_no": ""}
    head_lines = [p.text.strip() for p in paragraphs[:12] if p.text.strip()]
    for line in head_lines:
        m = re.search(r"หมายเลขเอกสาร\s*:\s*(\S+)", line)
        if m:
            meta["doc_no"] = m.group(1)
    subject = ""
    for i, line in enumerate(head_lines):
        if "preventive maintenance" in line.lower() and i + 1 < len(head_lines):
            subject = head_lines[i + 1]
            break
    if not subject:
        subject = Path(path).stem

    # --- personnel table -> responsible_lines ---
    responsible_lines = []
    for tbl in doc.tables:
        header = tbl.rows[0].cells[0].text.strip()
        if header == "บุคลากร" and len(tbl.rows) > 2:
            pos_cell = tbl.rows[2].cells[0].text
            duty_cell = tbl.rows[2].cells[1].text
            split_re = r"\n(?=\d+\.)|(?<!^)(?=\d+\.)"
            positions = [x.strip() for x in re.split(split_re, pos_cell) if x.strip()]
            duties = [x.strip() for x in re.split(split_re, duty_cell) if x.strip()]
            for pos, duty in zip(positions, duties):
                pos_clean = re.sub(r"\s+", " ", re.sub(r"^\d+\.\s*", "", pos)).strip()
                duty_clean = re.sub(r"\s+", " ", re.sub(r"^\d+\.\s*", "", duty)).strip()
                if pos_clean:
                    responsible_lines.append(f"{pos_clean}: {duty_clean}")
            break

    # --- safety precautions ("ข้อควรระวัง เพื่อความปลอดภัย") -> precaution_lines ---
    precaution_lines = []
    in_precautions = False
    for p in paragraphs:
        t = p.text.strip()
        if t == "ข้อควรระวัง เพื่อความปลอดภัย":
            in_precautions = True
            continue
        if in_precautions and (t.startswith("ระบุเครื่องมือ") or _CHAPTER_RE.match(t)):
            break
        if in_precautions and t and t not in ("ทั่วไป", "ข้อควรปฎิบัติ", "ข้อควรปฏิบัติ"):
            precaution_lines.append(t)

    # --- tools/materials/equipment ("1.3.1/1.3.2/1.3.3") -> equipment_list ---
    equipment_list = []
    section = None
    for p in paragraphs:
        t = p.text.strip()
        m = _CHAPTER_RE.match(t)
        if (m and m.group(1) != "1") or t == "รายละเอียดงาน":
            break
        if re.match(r"^1\.3\.1\b", t):
            section = "tools"
            continue
        if re.match(r"^1\.3\.2\b", t):
            section = "materials"
            continue
        if re.match(r"^1\.3\.3\b", t):
            section = "equipment"
            continue
        if section and t:
            equipment_list.append(re.sub(r"^-\s*", "", t).strip())

    # --- checklist summary table -> ordered (group, step_num, short_desc) ---
    steps_table = None
    for tbl in doc.tables:
        header = " ".join(c.text for c in tbl.rows[0].cells)
        if "รายละเอียดงาน" in header:
            steps_table = tbl
            break

    raw_items = []
    group_names = []
    current_group = None
    if steps_table is not None:
        for row in steps_table.rows[2:]:
            num = row.cells[0].text.strip()
            desc = row.cells[1].text.strip()
            if not num or not desc:
                continue
            if "." not in num:
                current_group = desc
                group_names.append(desc)
                continue
            raw_items.append((current_group, num, desc))

    # --- body scan: find "บทที่ 2", then walk each step's short description to
    # an anchor paragraph and harvest detail text + photos up to the next step ---
    chapter2_idx = None
    for i, p in enumerate(paragraphs):
        m = _CHAPTER_RE.search(p.text)
        if m and m.group(1) == "2":
            chapter2_idx = i
            break
    body_paragraphs = paragraphs[chapter2_idx:] if chapter2_idx is not None else paragraphs
    group_names_norm = {_norm_step_text(g) for g in group_names}

    def find_anchor(target_norm, start):
        for j in range(start, len(body_paragraphs)):
            cand = _norm_step_text(body_paragraphs[j].text)
            if not cand or cand in group_names_norm:
                continue
            if cand == target_norm or (len(target_norm) > 8 and (target_norm in cand or cand in target_norm)):
                return j
        return None

    steps = []
    search_pos = 0
    for gi, (group, num, short) in enumerate(raw_items):
        target = _norm_step_text(short)
        anchor_idx = find_anchor(target, search_pos)
        if anchor_idx is None:
            steps.append({"detail": f"{num} {short}".strip(), "images": []})
            continue
        next_target = _norm_step_text(raw_items[gi + 1][2]) if gi + 1 < len(raw_items) else None
        end_idx = len(body_paragraphs)
        if next_target:
            found_end = find_anchor(next_target, anchor_idx + 1)
            if found_end is not None:
                end_idx = found_end
        detail_parts = [short]
        images = []
        for j in range(anchor_idx + 1, end_idx):
            para = body_paragraphs[j]
            t = para.text.strip()
            if CAPTION_START_RE.match(t):
                continue
            if _norm_step_text(t) in group_names_norm:
                continue
            images.extend(_paragraph_images(para._p, doc_part))
            if t and _is_meaningful_step(t):
                detail_parts.append(t)
        steps.append({
            "detail": re.sub(r"\s+", " ", " ".join(detail_parts)).strip(),
            "images": images,
        })
        search_pos = anchor_idx + 1

    groups = [(None, []), ("6M", steps)]

    return {
        "meta": {"doc_no": meta["doc_no"], "subject": subject, "title": subject},
        "purpose_lines": [],
        "responsible_lines": responsible_lines,
        "precaution_lines": precaution_lines,
        "equipment_list": equipment_list,
        "groups": groups,
    }


def _looks_like_chapter_format(doc):
    for p in doc.paragraphs[:20]:
        if _CHAPTER_RE.match(p.text.strip()):
            return True
    return False


def _looks_like_banner_format(doc):
    for tbl in doc.tables:
        if len(tbl.rows) == 1 and tbl.rows[0].cells[0].text.strip() in SECTION_BANNERS.values():
            return True
    return False


def extract(path):
    """Auto-detect which source-WI layout `path` uses and parse it -- both
    parsers return the same shape (meta/purpose_lines/responsible_lines/
    precaution_lines/equipment_list/groups) regardless of source format."""
    doc = docx.Document(path)
    if _looks_like_chapter_format(doc):
        return _extract_chapter_format(path)
    return _extract_banner_format(path)


## 4. โค้ดแนะนำอุปกรณ์/เครื่องมือต่อขั้นตอน (`suggest_equipment`)

ไฟล์ต้นฉบับใหม่มีรายการอุปกรณ์จริงของงานนี้ (หมวด `อุปกรณ์`) ให้ใช้อ้างอิง โค้ดนี้จะ:
1. ตรวจก่อนว่าขั้นตอนนั้นเอ่ยชื่ออุปกรณ์ในรายการตรง ๆ หรือไม่ (เชื่อถือได้สูงสุด)
2. ถ้าไม่เจอ ใช้กติกาคำสำคัญเล็ก ๆ จับคู่กับอุปกรณ์ที่มีอยู่จริงในรายการ (เช่น "น็อต/สกรู" → ไขควง/ประแจ)
3. ถ้ายังไม่เจอ ใช้ค่า default "ตรวจสอบด้วยสายตา (ไม่ใช้เครื่องมือพิเศษ)"

เป็นการแนะนำแบบฮิวริสติกเช่นเดียวกับโค้ดเดิม — ควรให้ทีมงานตรวจทานอีกครั้งก่อนใช้จริง

In [ ]:
# ---------------------------------------------------------------------------
# 2. Equipment suggestion per step (grounded in the doc's own equipment list)
# ---------------------------------------------------------------------------

_FALLBACK_RULES = [
    (r"หกเหลี่ยม", ["ชุดประแจหกเหลี่ยม"]),
    (r"ประแจล็อค|ล็อคน็อต", ["ประแจล็อค"]),
    (r"ไขควง", ["ชุดไขควงหกแฉก"]),
    (r"น็อต|สกรู|ขัน", ["ชุดไขควงหกแฉก", "ชุดประแจหกเหลี่ยม", "ประแจล็อค"]),
    (r"สเปรย์|คราบ|ฝุ่น", ["สเปรย์ทำความสะอาดโลหะ", "ผ้าทำความสะอาด"]),
    (r"แปรง", ["แปรง"]),
    (r"ผ้า", ["ผ้าทำความสะอาด"]),
    (r"สว่าน|เจาะ", ["สว่านไฟฟ้า"]),
    (r"จัดเตรียมอุปกรณ์|เตรียมเครื่องมือ", ["ALL"]),
    (r"กล่องใส่อุปกรณ์", ["กล่องใส่อุปกรณ์เปล่า"]),
]
_DEFAULT_EQUIPMENT = "ตรวจสอบด้วยสายตา"


def suggest_equipment(detail_text, equipment_list):
    """Match a step's description against the WI's own equipment list first
    (direct name mentions are the strongest signal), falling back to a small
    keyword table, then a generic default. Heuristic -- review before use."""
    direct = [item for item in equipment_list if item and item in detail_text]
    if direct:
        return " / ".join(dict.fromkeys(direct))

    for pattern, names in _FALLBACK_RULES:
        if not re.search(pattern, detail_text):
            continue
        if names == ["ALL"]:
            return " / ".join(equipment_list) if equipment_list else _DEFAULT_EQUIPMENT
        # only suggest a name if it's actually one of THIS system's real
        # equipment items -- these keyword rules were tuned against the
        # AEP-system vocabulary and shouldn't leak into other systems whose
        # equipment list doesn't contain them
        picked = [n for n in names if n in equipment_list]
        if picked:
            return " / ".join(picked)

    return _DEFAULT_EQUIPMENT


## 5. โค้ดสร้างเอกสาร Work Instruction (`generate`)

รับไฟล์ WI ต้นฉบับได้ทั้งไฟล์เดียวหรือหลายไฟล์ (`source_wi_paths` เป็น list) แล้วรวมเป็นเอกสารผลลัพธ์เดียว โดยหัวข้อ 5 จะมีหัวข้อย่อย **5.n ชื่อระบบ** หนึ่งหัวข้อต่อไฟล์ต้นฉบับหนึ่งไฟล์ (เรียงตามลำดับไฟล์ที่อัปโหลด) ตามด้วยตาราง PM ของระบบนั้น ๆ (แยกตารางแต่ละรอบ PM คนละหน้าพร้อม Repeat Header Rows, แถบสีน้ำเงินที่แถวบอกรอบ, สีสลับแถวข้อมูล, เลขลำดับขั้นตอนต่อรอบ, ขั้นตอนเตรียมงานร่วมของระบบนั้นซ้ำไว้บนสุดของทุกตาราง PM ของระบบนั้น), คอลัมน์ **รูปภาพประกอบ** ที่แทรกรูปจริงจากไฟล์ต้นฉบับของระบบนั้น (ถ้าขั้นตอนนั้นมีรูป ไม่งั้นคงข้อความ placeholder เดิมไว้) และคอลัมน์ **อุปกรณ์/เครื่องมือที่ใช้** ที่ดึงจาก `suggest_equipment` โดยใช้รายการอุปกรณ์ของระบบนั้นเท่านั้น (ไม่ปนกับระบบอื่น)

หัวข้ออื่น ๆ ที่รวมข้อมูลจากทุกระบบ: หัวข้อ 1 (วัตถุประสงค์) และ 4 (หน้าที่และความรับผิดชอบ) รวมข้อความที่ไม่ซ้ำจากทุกไฟล์, หัวข้อ 6 (อุปกรณ์) แสดงรายการอุปกรณ์แยกเป็นกลุ่มตามชื่อระบบ, หัวข้อ 8 (เอกสารอ้างอิง), 9 (แบบฟอร์มที่ใช้) และ 10 (บันทึกที่จัดเก็บ) มีหนึ่งรายการ/แถวต่อระบบ

In [ ]:
"""
Generate a single Work-Instruction (.docx) document that can combine MULTIPLE
narrative-style source WI files (e.g. AEP system, OCR System, Hot stamp
System) into one output, with section 5 split into a numbered subsection
(5.1, 5.2, 5.3, ...) per system:
  - one or more source WI .docx files (each carrying its own embedded photos
    and equipment list) -> data source, one per machine/system
  - a Work Instruction template (.docx)                    -> layout/format to keep 100%

Usage:
    python generate_wi.py <template.docx> <output.docx> <source_wi_1.docx> [<source_wi_2.docx> ...]
"""
import copy
import io
import sys
from pathlib import Path

import docx
from docx.enum.text import WD_ALIGN_PARAGRAPH
from docx.oxml import OxmlElement, parse_xml
from docx.oxml.ns import nsdecls, qn
from docx.shared import Cm, Pt, RGBColor
from docx.table import Table
from docx.text.paragraph import Paragraph


IMAGE_WIDTH_CM = 5.5
BODY_FONT_NAME = "TH Sarabun New"
BODY_FONT_SIZE = Pt(16)
EQUIPMENT_FONT_SIZE = Pt(14)


def _set_run_font(run, name=BODY_FONT_NAME, size=BODY_FONT_SIZE):
    run.font.name = name
    run.font.size = size
    rPr = run._r.get_or_add_rPr()
    rFonts = rPr.find(qn("w:rFonts"))
    if rFonts is None:
        rFonts = OxmlElement("w:rFonts")
        rPr.append(rFonts)
    rFonts.set(qn("w:ascii"), name)
    rFonts.set(qn("w:hAnsi"), name)
    rFonts.set(qn("w:cs"), name)
    # Thai text is complex-script, sized via w:szCs -- python-docx's
    # font.size setter only writes w:sz (the Latin size), so without this
    # Thai glyphs fell back to whatever szCs the run/style happened to have
    # (or its default), ignoring the size we just set. Same family of bug
    # as the w:bCs/w:iCs cases already fixed elsewhere.
    szCs = rPr.find(qn("w:szCs"))
    if szCs is None:
        szCs = OxmlElement("w:szCs")
        rPr.append(szCs)
    szCs.set(qn("w:val"), str(int(size.pt * 2)))


def set_paragraph_text(paragraph, text):
    if not paragraph.runs:
        paragraph.add_run(text)
        return
    first = paragraph.runs[0]
    first.text = text
    for extra in paragraph.runs[1:]:
        extra.text = ""


def normalize_run_formatting(paragraph, size=BODY_FONT_SIZE):
    """Force every non-empty run in `paragraph` to plain black, non-bold --
    without touching its text. Used for template instructional/example text
    we leave in place (section 3's definitions example, the amendment
    record's sample row, ...) that still carries the template's gray/bold
    "placeholder" styling."""
    for run in paragraph.runs:
        if run.text.strip():
            run.font.color.rgb = RGBColor(0, 0, 0)
            run.font.bold = False
            # Thai text renders via the complex-script ("Cs") run properties.
            # A bare <w:bCs/> with no val defaults to "on", so Thai glyphs
            # stayed bold even with <w:b val="0"/> correctly turning off the
            # Latin flag -- same trap as the w:iCs italic issue elsewhere.
            rPr = run._r.find(qn("w:rPr"))
            if rPr is not None:
                bCs = rPr.find(qn("w:bCs"))
                if bCs is not None:
                    bCs.set(qn("w:val"), "0")
            # The template's placeholder paragraphs keep their real text in
            # runs AFTER the first (tab-only) run, which is the one whose
            # text set_paragraph_text actually overwrites -- and that first
            # run has no explicit size, so it silently fell back to the
            # document's un-styled default (12pt) instead of the 16pt TH
            # Sarabun New every other visible run in the file uses.
            _set_run_font(run, size=size)


def set_content_text(paragraph, text, size=BODY_FONT_SIZE):
    """Like set_paragraph_text, but also normalizes formatting: many of the
    template's placeholder runs ("ระบุ...", "(หากเลือกใช้...)") are styled
    gray/bold to signal "fill this in" -- once real generated content lands
    there it should read as plain black body text, not inherit that look."""
    set_paragraph_text(paragraph, text)
    normalize_run_formatting(paragraph, size=size)


def normalize_until_next_heading(doc, start_index):
    """Normalize every paragraph from start_index up to (not including) the
    next Heading-1 paragraph -- for template instructional text that sits
    between a section heading and the next one, which we never overwrite
    but still shouldn't look grayed-out."""
    for p in doc.paragraphs[start_index:]:
        if p.style.name == "Heading 1":
            break
        normalize_run_formatting(p)


def find_paragraph(doc, predicate):
    for i, p in enumerate(doc.paragraphs):
        if predicate(p.text):
            return i
    return None


def find_heading(doc, prefix):
    for i, p in enumerate(doc.paragraphs):
        if p.style.name == "Heading 1" and p.text.strip().startswith(prefix):
            return i
    return None


def clone_paragraph_after(paragraph, new_text, content=False):
    new_p_elem = copy.deepcopy(paragraph._p)
    paragraph._p.addnext(new_p_elem)
    new_para = Paragraph(new_p_elem, paragraph._parent)
    (set_content_text if content else set_paragraph_text)(new_para, new_text)
    return new_para


def _is_blank_paragraph_elem(p_elem):
    return p_elem is not None and p_elem.tag == qn("w:p") and not "".join(p_elem.xpath(".//w:t/text()")).strip()


def remove_next_blank_paragraph(paragraph):
    nxt = paragraph._p.getnext()
    if _is_blank_paragraph_elem(nxt):
        nxt.getparent().remove(nxt)


def trim_blank_paragraphs_before(elem, keep):
    blanks = []
    prev = elem.getprevious()
    while _is_blank_paragraph_elem(prev):
        blanks.append(prev)
        prev = prev.getprevious()
    for extra in blanks[keep:]:
        extra.getparent().remove(extra)


def insert_page_break_before(elem):
    new_p = OxmlElement("w:p")
    run = OxmlElement("w:r")
    br = OxmlElement("w:br")
    br.set(qn("w:type"), "page")
    run.append(br)
    new_p.append(run)
    elem.addprevious(new_p)


_HEADING2_STYLE_XML = """
<w:style {nsdecls} w:type="paragraph" w:styleId="Heading2">
  <w:name w:val="heading 2"/>
  <w:basedOn w:val="Normal"/>
  <w:next w:val="Normal"/>
  <w:link w:val="Heading2Char"/>
  <w:qFormat/>
  <w:pPr>
    <w:keepNext/>
    <w:spacing w:before="240" w:after="60"/>
    <w:outlineLvl w:val="1"/>
  </w:pPr>
  <w:rPr>
    <w:rFonts w:ascii="{font}" w:hAnsi="{font}" w:cs="{font}"/>
    <w:b/>
    <w:bCs/>
    <w:sz w:val="{sz}"/>
    <w:szCs w:val="{sz}"/>
  </w:rPr>
</w:style>
"""

_HEADING2_CHAR_STYLE_XML = """
<w:style {nsdecls} w:type="character" w:customStyle="1" w:styleId="Heading2Char">
  <w:name w:val="Heading 2 Char"/>
  <w:link w:val="Heading2"/>
  <w:rPr>
    <w:rFonts w:ascii="{font}" w:hAnsi="{font}" w:cs="{font}"/>
    <w:b/>
    <w:bCs/>
    <w:sz w:val="{sz}"/>
    <w:szCs w:val="{sz}"/>
  </w:rPr>
</w:style>
"""


def ensure_heading2_style(doc):
    """The template only ships a 'Heading 1' style. Build a real, built-in-
    shaped 'Heading 2' (styleId "Heading2", linked char style, qFormat,
    outline level 1) -- not just a custom style that happens to be named
    "Heading 2" -- so Word's Quick Styles gallery highlights it correctly
    and the document's { TOC \\o "1-3" } field picks up the '5.n <system>'
    subsection headings on refresh."""
    for s in doc.styles:
        if s.name == "Heading 2":
            return s
    sz = str(int(BODY_FONT_SIZE.pt * 2))  # half-points
    styles_elm = doc.styles.element
    styles_elm.append(parse_xml(
        _HEADING2_CHAR_STYLE_XML.format(nsdecls=nsdecls("w"), font=BODY_FONT_NAME, sz=sz)
    ))
    styles_elm.append(parse_xml(
        _HEADING2_STYLE_XML.format(nsdecls=nsdecls("w"), font=BODY_FONT_NAME, sz=sz)
    ))
    return doc.styles["Heading 2"]


def insert_subheading_before(elem, parent, heading2_style, text):
    """Insert a '5.n <system name>' subsection heading, styled 'Heading 2',
    right before `elem`."""
    new_p = OxmlElement("w:p")
    elem.addprevious(new_p)
    new_para = Paragraph(new_p, parent)
    new_para.style = heading2_style
    new_para.add_run(text)
    return new_para


BORDER_EDGES = ("top", "left", "bottom", "right")


def set_cell_borders(cell, **edges):
    tcPr = cell._tc.get_or_add_tcPr()
    tcBorders = tcPr.find(qn("w:tcBorders"))
    if tcBorders is None:
        tcBorders = OxmlElement("w:tcBorders")
        tcPr.append(tcBorders)
    for edge in BORDER_EDGES:
        spec = edges.get(edge)
        if spec is None:
            continue
        tag = qn(f"w:{edge}")
        elem = tcBorders.find(tag)
        if elem is None:
            elem = OxmlElement(f"w:{edge}")
            tcBorders.append(elem)
        elem.set(qn("w:val"), spec.get("val", "single"))
        if "sz" in spec:
            elem.set(qn("w:sz"), str(spec["sz"]))
        if "color" in spec:
            elem.set(qn("w:color"), spec["color"])


def set_cell_shading(cell, hex_color):
    tcPr = cell._tc.get_or_add_tcPr()
    shd = tcPr.find(qn("w:shd"))
    if shd is None:
        shd = OxmlElement("w:shd")
        tcPr.append(shd)
    shd.set(qn("w:val"), "clear")
    shd.set(qn("w:color"), "auto")
    shd.set(qn("w:fill"), hex_color)


def clear_run_italic(run):
    run.font.italic = False
    rPr = run._r.find(qn("w:rPr"))
    if rPr is not None:
        iCs = rPr.find(qn("w:iCs"))
        if iCs is not None:
            iCs.set(qn("w:val"), "0")


def set_table_header_row(row):
    tr = row._tr
    trPr = tr.find(qn("w:trPr"))
    if trPr is None:
        trPr = OxmlElement("w:trPr")
        tr.insert(0, trPr)
    if trPr.find(qn("w:tblHeader")) is None:
        trPr.insert(0, OxmlElement("w:tblHeader"))


def clear_row_height(row):
    trPr = row._tr.find(qn("w:trPr"))
    if trPr is not None:
        h = trPr.find(qn("w:trHeight"))
        if h is not None:
            trPr.remove(h)


def set_cell_vertical_margin(cell, top_twips, bottom_twips):
    tcPr = cell._tc.get_or_add_tcPr()
    tcMar = tcPr.find(qn("w:tcMar"))
    if tcMar is None:
        tcMar = OxmlElement("w:tcMar")
        tcPr.append(tcMar)
    for edge, value in (("top", top_twips), ("bottom", bottom_twips)):
        elem = tcMar.find(qn(f"w:{edge}"))
        if elem is None:
            elem = OxmlElement(f"w:{edge}")
            tcMar.append(elem)
        elem.set(qn("w:w"), str(value))
        elem.set(qn("w:type"), "dxa")


def find_table(doc, header_contains):
    for tbl in doc.tables:
        if header_contains in " ".join(c.text for c in tbl.rows[0].cells):
            return tbl
    return None


def append_row_like(table, template_tr):
    """Append a new row (cloned from template_tr) at the end of `table`."""
    new_tr = copy.deepcopy(template_tr)
    table._tbl.append(new_tr)
    return table.rows[-1]


def set_cell_pictures(cell, image_blobs):
    """Embed the step's real photo(s) in the 'รูปภาพประกอบ' cell, one per
    paragraph; falls back to the template's placeholder text when the
    source document had no photo for this step."""
    first_para = cell.paragraphs[0]
    set_paragraph_text(first_para, "")
    if not image_blobs:
        set_paragraph_text(first_para, "(แนบรูปภาพ / ภาพร่าง)")
        for run in first_para.runs:
            if run.text.strip():
                run.font.color.rgb = RGBColor(0, 0, 0)
        return
    first_para.alignment = WD_ALIGN_PARAGRAPH.CENTER
    first_para.add_run().add_picture(io.BytesIO(image_blobs[0]), width=Cm(IMAGE_WIDTH_CM))
    for blob in image_blobs[1:]:
        p = cell.add_paragraph()
        p.alignment = WD_ALIGN_PARAGRAPH.CENTER
        p.add_run().add_picture(io.BytesIO(blob), width=Cm(IMAGE_WIDTH_CM))


def _fill_one_cycle_table(anchor, parent, purpose_tr_template, header_tr, item_tr_template,
                           purpose_text, cycle, all_steps, equipment_list):
    new_tbl_elem = copy.deepcopy(anchor)
    for tr in list(new_tbl_elem.findall(qn("w:tr"))):
        new_tbl_elem.remove(tr)
    new_tbl_elem.append(copy.deepcopy(purpose_tr_template))
    new_tbl_elem.append(copy.deepcopy(header_tr))
    new_tbl_elem.append(copy.deepcopy(item_tr_template))  # cycle label row
    for _ in all_steps:
        new_tbl_elem.append(copy.deepcopy(item_tr_template))

    anchor.addprevious(new_tbl_elem)
    new_table = Table(new_tbl_elem, parent)

    set_paragraph_text(new_table.rows[0].cells[0].paragraphs[0], f"จุดประสงค์ (PURPOSE): {purpose_text}")
    set_table_header_row(new_table.rows[0])
    set_table_header_row(new_table.rows[1])

    label_row = new_table.rows[2]
    set_table_header_row(label_row)
    set_paragraph_text(label_row.cells[0].paragraphs[0], "")
    set_paragraph_text(label_row.cells[1].paragraphs[0], CYCLE_LABELS.get(cycle, cycle))
    set_paragraph_text(label_row.cells[2].paragraphs[0], "")
    set_paragraph_text(label_row.cells[3].paragraphs[0], "")
    label_row.cells[1].paragraphs[0].alignment = WD_ALIGN_PARAGRAPH.LEFT
    if label_row.cells[1].paragraphs[0].runs:
        label_row.cells[1].paragraphs[0].runs[0].bold = True
    clear_row_height(label_row)

    blue = {"val": "single", "sz": "4", "color": "2E75B6"}
    nil = {"val": "nil"}
    set_cell_borders(label_row.cells[0], top=blue, bottom=blue, left=blue, right=nil)
    set_cell_borders(label_row.cells[1], top=blue, bottom=blue, left=nil, right=nil)
    set_cell_borders(label_row.cells[2], top=blue, bottom=blue, left=nil, right=nil)
    set_cell_borders(label_row.cells[3], top=blue, bottom=blue, left=nil, right=blue)

    for step_num, item in enumerate(all_steps, start=1):
        row = new_table.rows[step_num + 2]
        set_content_text(row.cells[0].paragraphs[0], str(step_num))
        set_content_text(row.cells[1].paragraphs[0], item["detail"])
        set_cell_pictures(row.cells[2], item.get("images", []))

        equipment_text = suggest_equipment(item["detail"], equipment_list)
        set_content_text(row.cells[3].paragraphs[0], equipment_text, size=EQUIPMENT_FONT_SIZE)
        clear_run_italic(row.cells[3].paragraphs[0].runs[0])

        band_color = "FFFFFF" if step_num % 2 == 1 else "DEEAF1"
        for cell in row.cells:
            set_cell_shading(cell, band_color)

    for row in new_table.rows:
        for cell in row.cells:
            set_cell_vertical_margin(cell, top_twips=40, bottom_twips=40)


def fill_steps_section(doc, table, systems):
    """Build section 5 as one '5.n <system name>' subsection per system, each
    followed by that system's one-table-per-inspection-cycle tables (with the
    system's own common prep steps repeated at the top of every cycle table)."""
    header_tr = copy.deepcopy(table.rows[1]._tr)
    purpose_tr_template = copy.deepcopy(table.rows[0]._tr)
    item_tr_template = copy.deepcopy(table.rows[2]._tr)
    parent = table._parent
    anchor = table._tbl
    heading2_style = ensure_heading2_style(doc)

    trim_blank_paragraphs_before(anchor, keep=2)

    for sys_idx, system in enumerate(systems, start=1):
        if sys_idx > 1:
            insert_page_break_before(anchor)
        insert_subheading_before(anchor, parent, heading2_style, f"5.{sys_idx} {system['subject']}")

        for cycle_idx, (cycle, cycle_steps) in enumerate(system["cycle_groups"]):
            if cycle_idx > 0:
                insert_page_break_before(anchor)
            all_steps = system["preamble_steps"] + cycle_steps
            _fill_one_cycle_table(
                anchor, parent, purpose_tr_template, header_tr, item_tr_template,
                system["subject"], cycle, all_steps, system["equipment_list"],
            )

    anchor.getparent().remove(anchor)


def _dedup_lines(list_of_line_lists):
    out = []
    for lines in list_of_line_lists:
        for line in lines:
            if line not in out:
                out.append(line)
    return out


def generate(source_wi_paths, template_path, output_path):
    if isinstance(source_wi_paths, (str, Path)):
        source_wi_paths = [source_wi_paths]

    systems = []
    for p in source_wi_paths:
        d = extract(p)
        groups = d["groups"]
        preamble_steps = groups[0][1] if groups and groups[0][0] is None else []
        cycle_groups = [(c, steps) for c, steps in groups if c is not None]
        systems.append({
            "subject": d["meta"]["subject"] or Path(p).stem,
            "doc_code": d["meta"]["doc_no"] or Path(p).stem,
            "purpose_lines": d["purpose_lines"],
            "responsible_lines": d["responsible_lines"],
            "precaution_lines": d.get("precaution_lines", []),
            "equipment_list": d["equipment_list"],
            "preamble_steps": preamble_steps,
            "cycle_groups": cycle_groups,
        })

    subjects_str = " / ".join(s["subject"] for s in systems)

    doc = docx.Document(template_path)

    # --- Title page ---
    i = find_paragraph(doc, lambda t: t.strip() == "ชื่อเอกสาร")
    if i is not None:
        title_para = doc.paragraphs[i]
        trim_blank_paragraphs_before(title_para._p, keep=7)
        set_paragraph_text(title_para, "ขั้นตอนการตรวจสอบและบำรุงรักษาเครื่องจักรเชิงป้องกัน ")
        clone_paragraph_after(title_para, subjects_str)

    i = find_paragraph(doc, lambda t: t.strip().startswith("หน่วยงาน"))
    if i is not None:
        dept_para = doc.paragraphs[i]
        trim_blank_paragraphs_before(dept_para._p, keep=6)
        set_paragraph_text(dept_para, "หน่วยงาน แผนกปฏิบัติการและซ่อมบำรุง (Operation & Maintenance Department)")

    i = find_paragraph(doc, lambda t: t.strip() == "สารบัญ")
    if i is not None:
        insert_page_break_before(doc.paragraphs[i]._p)

    # --- 1. Purpose (union of each system's own วัตถุประสงค์ text) ---
    i = find_heading(doc, "1. วัตถุประสงค์")
    if i is not None:
        content_para = doc.paragraphs[i + 1]
        purpose_lines = _dedup_lines(s["purpose_lines"] for s in systems)
        if purpose_lines:
            text = "\n".join(f"\t{line}" for line in purpose_lines)
        else:
            text = (
                f"\tเพื่อกำหนดขั้นตอนการตรวจสอบและบำรุงรักษาเชิงป้องกัน (Preventive Maintenance) ของเครื่องจักร {subjects_str} "
                f"ให้เป็นไปตามรอบเวลาที่กำหนด เพื่อให้เครื่องจักรทำงานได้อย่างมีประสิทธิภาพ ปลอดภัย "
                f"และลดโอกาสการหยุดทำงานกะทันหัน"
            )
        set_content_text(content_para, text)
        remove_next_blank_paragraph(content_para)

    # --- 2. Scope ---
    i = find_heading(doc, "2. ขอบเขต")
    if i is not None:
        cycles = sorted(
            {c for s in systems for c, _ in s["cycle_groups"]},
            key=lambda c: list(CYCLE_LABELS).index(c) if c in CYCLE_LABELS else 99,
        )
        cycle_str = " / ".join(CYCLE_LABELS.get(c, c) for c in cycles)
        systems_str = ", ".join(f"{s['subject']} ({s['doc_code']})" for s in systems)
        content_para = doc.paragraphs[i + 1]
        set_content_text(
            content_para,
            f"\tครอบคลุมขั้นตอนการตรวจสอบและบำรุงรักษาระบบ {systems_str} "
            f"ตามรอบการตรวจสอบ {cycle_str}",
        )
        remove_next_blank_paragraph(content_para)

    # --- 3. Definitions -- left as the template's own example (no P/F legend
    # in the new source format), just de-grayed/de-bolded like everything else ---
    i = find_heading(doc, "3.")
    if i is not None:
        normalize_until_next_heading(doc, i + 1)

    # --- 4. Responsibilities (union of each system's own ผู้ปฏิบัติ text) ---
    i = find_heading(doc, "4.  หน้าที่และความรับผิดชอบ")
    if i is not None:
        content_para = doc.paragraphs[i + 1]
        responsible_lines = _dedup_lines(s["responsible_lines"] for s in systems)
        if responsible_lines:
            text = "\n".join(f"\t{line}" for line in responsible_lines)
        else:
            text = (
                "\tวิศวกร/ช่างเทคนิค: ดำเนินการตรวจสอบและบันทึกผลตามขั้นตอนที่กำหนด\n"
                "\tวิศวกรผู้ควบคุมงาน: ตรวจสอบความถูกต้องและอนุมัติผลการตรวจสอบ"
            )
        set_content_text(content_para, text)
        remove_next_blank_paragraph(content_para)

    # --- 5. Work steps: one '5.n <system>' subsection per system ---
    heading_i = find_heading(doc, "5.  ขั้นตอนการปฏิบัติงาน")
    steps_table = find_table(doc, "จุดประสงค์")
    if heading_i is not None:
        intro_para = doc.paragraphs[heading_i + 1]
        set_content_text(
            intro_para,
            "\tอธิบายขั้นตอนการทำงานอย่างละเอียดและเป็นลำดับขั้นตอน เข้าใจง่าย "
            "และสามารถปฏิบัติตามได้จริง รวมถึงภาพประกอบ/สื่อช่วย (ถ้ามี)",
        )
        # drop the template's "2 formats" list (text/table) note entirely
        for p in list(doc.paragraphs):
            t = p.text.strip()
            if t.startswith("1) แบบบรรยายข้อความ") or t.startswith("2) แบบตารางที่มีช่องใส่รูปภาพ"):
                p._p.getparent().remove(p._p)
    if steps_table is not None and heading_i is not None:
        fill_steps_section(doc, steps_table, systems)

    # --- 6. Equipment (each system's own extracted equipment list) ---
    # Each system's equipment list is used to fill in section 5's own
    # "อุปกรณ์/เครื่องมือที่ใช้" column per step (see suggest_equipment) --
    # section 6 just points there instead of repeating the list.
    i = find_heading(doc, "6.  อุปกรณ์")
    if i is not None:
        set_content_text(doc.paragraphs[i + 1], "\tตามที่ระบุในข้อ 5 (คอลัมน์ \"อุปกรณ์/เครื่องมือที่ใช้\" ของแต่ละขั้นตอน)")

    # --- 7. Precautions (union of each system's own precaution text, if any) ---
    i = find_heading(doc, "7.  ข้อควรระวัง")
    if i is not None:
        precaution_lines = _dedup_lines(s["precaution_lines"] for s in systems)
        if precaution_lines:
            text = "\n".join(f"\t{line}" for line in precaution_lines)
        else:
            text = (
                "\tปิดเครื่องจักรและตัดแหล่งพลังงานก่อนตรวจสอบ/บำรุงรักษาทุกครั้ง สวมใส่อุปกรณ์ป้องกันความปลอดภัยส่วนบุคคล (PPE) "
                "ตลอดเวลาปฏิบัติงาน ระมัดระวังชิ้นส่วนที่เคลื่อนที่และระบบไฮดรอลิกที่มีแรงดันสูง "
                "หากพบความผิดปกติให้หยุดปฏิบัติงานและแจ้งวิศวกรผู้ควบคุมงานทันที"
            )
        set_content_text(doc.paragraphs[i + 1], text)

    # --- 8. References (one line per system) ---
    i = find_heading(doc, "8.  เอกสารอ้างอิง")
    if i is not None:
        text = "\n".join(f"\tแบบฟอร์ม Check Sheet เลขที่ {s['doc_code']} ({s['subject']})" for s in systems)
        set_content_text(doc.paragraphs[i + 1], text)

    # --- 9. Forms used (9.1, 9.2, ... one per system) ---
    heading_i = find_heading(doc, "9.  แบบฟอร์มที่ใช้")
    if heading_i is not None:
        normalize_run_formatting(doc.paragraphs[heading_i + 1])  # the template's own intro sentence
    i = find_paragraph(doc, lambda t: t.strip().startswith("9.1"))
    if i is not None:
        last_para = doc.paragraphs[i]
        set_content_text(last_para, f"\t9.1 ใบบันทึกผลตรวจสอบการทำงาน {systems[0]['subject']} ({systems[0]['doc_code']})")
        for idx, s in enumerate(systems[1:], start=2):
            last_para = clone_paragraph_after(
                last_para, f"\t9.{idx} ใบบันทึกผลตรวจสอบการทำงาน {s['subject']} ({s['doc_code']})", content=True
            )

    # --- 10. Records retained (one row per system) ---
    heading_i = find_heading(doc, "10.  บันทึกที่จัดเก็บ")
    if heading_i is not None:
        normalize_run_formatting(doc.paragraphs[heading_i + 1])  # the template's own intro sentence
    records_table = find_table(doc, "ชื่อบันทึกที่จัดเก็บ")
    if records_table is not None:
        row_template_tr = copy.deepcopy(records_table.rows[1]._tr)
        if len(records_table.rows) > 2:
            records_table._tbl.remove(records_table.rows[2]._tr)  # drop the template's spare blank row
        for idx, s in enumerate(systems):
            row = records_table.rows[1] if idx == 0 else append_row_like(records_table, row_template_tr)
            cells = row.cells
            set_content_text(cells[0].paragraphs[0], str(idx + 1))
            set_content_text(cells[1].paragraphs[0], f"ใบบันทึกผลตรวจสอบการทำงาน {s['subject']} ({s['doc_code']})")
            set_content_text(cells[2].paragraphs[0], "แผนกซ่อมบำรุง")
            set_content_text(cells[3].paragraphs[0], "3 ปี")
            set_content_text(cells[4].paragraphs[0], "แฟ้ม")
            set_content_text(cells[5].paragraphs[0], "ตู้เอกสารแผนกซ่อมบำรุง")
            set_content_text(cells[6].paragraphs[0], "ผู้อำนวยการฝ่าย")

    i = find_paragraph(doc, lambda t: "บันทึกการแก้ไข" in t)
    if i is not None:
        trim_blank_paragraphs_before(doc.paragraphs[i]._p, keep=2)

    # the amendment-record table's own example row is left as template
    # boilerplate (we don't generate real history here) -- just de-gray it
    amendment_table = find_table(doc, "รายละเอียดการแก้ไข")
    if amendment_table is not None and len(amendment_table.rows) > 1:
        for cell in amendment_table.rows[1].cells:
            normalize_run_formatting(cell.paragraphs[0])

    doc.save(output_path)
    return systems


## 6. รันสร้างเอกสาร แล้วดาวน์โหลดผลลัพธ์

In [ ]:
output_path = "WI_output.docx"
systems = generate(source_wi_paths, template_path, output_path)

print(f"สร้างสำเร็จ: {output_path}")
print(f"จำนวนระบบที่รวมในเอกสาร: {len(systems)}")
for idx, s in enumerate(systems, start=1):
    n_steps = sum(len(steps) for _, steps in s["cycle_groups"]) + len(s["preamble_steps"]) * len(s["cycle_groups"])
    n_images = sum(len(st["images"]) for _, steps in s["cycle_groups"] for st in steps) + \
        sum(len(st["images"]) for st in s["preamble_steps"]) * len(s["cycle_groups"])
    cycles = sorted({c for c, _ in s["cycle_groups"]})
    print(f"  5.{idx} {s['subject']} ({s['doc_code']}): {n_steps} ขั้นตอนรวม, {n_images} รูปที่ฝัง, รอบ {', '.join(cycles)}")
    print(f"       อุปกรณ์: {', '.join(s['equipment_list']) or '(ไม่พบ)'}")

files.download(output_path)